# 5.1 Speech Trial 001: MFCC + Random Forest

Purpose:
- Train MVP speech baseline
- Use speaker-safe splits
- Evaluate dysarthric vs control classification

In [ ]:
import numpy as np
import pandas as pd

from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

from rural_stroke_assist.config import (
    SPEECH_FEATURES_PATH,
    SPEECH_TRIAL_001_MODEL_PATH,
    SPEECH_TRIAL_001_RESULTS_PATH,
    SPEECH_TRIAL_001_PREDICTIONS_PATH,
    SPEECH_TRIAL_001_CONFUSION_MATRIX_PATH,
)
from rural_stroke_assist.features.speech_features import SPEECH_INVERSE_LABEL_MAPPING
from rural_stroke_assist.modeling.speech_baselines import (
    get_speech_feature_columns,
    build_random_forest_speech_pipeline,
    save_model,
)
from rural_stroke_assist.modeling.evaluation import (
    evaluate_binary_classifier,
    create_classification_report_dict,
    save_confusion_matrix_plot,
    save_markdown_results,
)

In [3]:
speech_features_df = pd.read_csv(SPEECH_FEATURES_PATH)

speech_features_df.head()

,path,file_name,label,label_encoded,speaker_id,torgo_group,split,mfcc_1_mean,mfcc_2_mean,mfcc_3_mean,...,rms_mean,rms_std,zcr_mean,zcr_std,spectral_centroid_mean,spectral_centroid_std,spectral_bandwidth_mean,spectral_bandwidth_std,spectral_rolloff_mean,spectral_rolloff_std
0,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,wav_arrayMic_M01S01_0001.wav,dysarthric,1,wav_arrayMic_M01S01,M_Dys,train,-163.494843,56.581371,-30.107115,...,0.144104,0.102247,0.151883,0.121665,2042.334292,781.147275,1723.756351,442.497088,3784.583997,1448.862083
1,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,wav_arrayMic_M01S01_0002.wav,dysarthric,1,wav_arrayMic_M01S01,M_Dys,train,-196.675262,66.341858,-25.319544,...,0.108665,0.071716,0.154005,0.139137,1910.950448,865.713928,1632.424714,357.816673,3471.785430,1342.273322
2,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,wav_arrayMic_M01S01_0003.wav,dysarthric,1,wav_arrayMic_M01S01,M_Dys,train,-181.038422,52.951725,-29.189020,...,0.132562,0.098360,0.232347,0.199550,2342.658893,1164.214791,1703.955806,427.230919,3979.150080,1727.665199
3,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,wav_arrayMic_M01S01_0004.wav,dysarthric,1,wav_arrayMic_M01S01,M_Dys,train,-376.797882,17.045906,36.214321,...,0.007747,0.007332,0.417319,0.225873,3702.937939,1098.628999,2367.278508,269.317380,6438.893312,978.800780
4,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,wav_arrayMic_M01S01_0005.wav,dysarthric,1,wav_arrayMic_M01S01,M_Dys,train,-226.536652,40.652927,-18.713095,...,0.097521,0.098568,0.229228,0.181667,2410.353555,1014.842146,1835.126339,420.014444,4338.126990,1615.570376


In [4]:
speech_features_df.groupby(["split", "label"]).size()

split  label     
test   control       1867
       dysarthric     982
train  control       6678
       dysarthric    4221
val    control       2909
       dysarthric     973
dtype: int64

In [5]:
feature_columns = get_speech_feature_columns(speech_features_df)

feature_columns[:10], len(feature_columns)

(['mfcc_1_mean',
  'mfcc_2_mean',
  'mfcc_3_mean',
  'mfcc_4_mean',
  'mfcc_5_mean',
  'mfcc_6_mean',
  'mfcc_7_mean',
  'mfcc_8_mean',
  'mfcc_9_mean',
  'mfcc_10_mean'],
 37)

In [6]:
train_df = speech_features_df[speech_features_df["split"] == "train"].copy()
val_df = speech_features_df[speech_features_df["split"] == "val"].copy()
test_df = speech_features_df[speech_features_df["split"] == "test"].copy()

X_train = train_df[feature_columns]
y_train = train_df["label_encoded"]

X_val = val_df[feature_columns]
y_val = val_df["label_encoded"]

X_test = test_df[feature_columns]
y_test = test_df["label_encoded"]

In [7]:
model = build_random_forest_speech_pipeline(random_state=42)

model.fit(X_train, y_train)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('model',
                 RandomForestClassifier(class_weight='balanced',
                                        n_estimators=300, n_jobs=-1,
                                        random_state=42))])

In [8]:
val_pred = model.predict(X_val)
val_score = model.predict_proba(X_val)[:, 1]

val_metrics = evaluate_binary_classifier(
    y_true=y_val.to_numpy(),
    y_pred=val_pred,
    y_score=val_score,
)

val_metrics

{'accuracy': 0.7284904688304997,
 'precision': 0.4321608040201005,
 'recall': 0.26515930113052416,
 'f1': 0.3286624203821656,
 'roc_auc': 0.5624658844843784}

In [9]:
test_pred = model.predict(X_test)
test_score = model.predict_proba(X_test)[:, 1]

test_metrics = evaluate_binary_classifier(
    y_true=y_test.to_numpy(),
    y_pred=test_pred,
    y_score=test_score,
)

test_metrics

{'accuracy': 0.8055458055458056,
 'precision': 0.6494413407821229,
 'recall': 0.9470468431771895,
 'f1': 0.7705053852526926,
 'roc_auc': 0.9426279894010781}

In [10]:
target_names = [
    SPEECH_INVERSE_LABEL_MAPPING[0],
    SPEECH_INVERSE_LABEL_MAPPING[1],
]

report = create_classification_report_dict(
    y_true=y_test.to_numpy(),
    y_pred=test_pred,
    target_names=target_names,
)

report

{'control': {'precision': 0.963302752293578,
  'recall': 0.7311194429566149,
  'f1-score': 0.8313032886723508,
  'support': 1867.0},
 'dysarthric': {'precision': 0.6494413407821229,
  'recall': 0.9470468431771895,
  'f1-score': 0.7705053852526926,
  'support': 982.0},
 'accuracy': 0.8055458055458056,
 'macro avg': {'precision': 0.8063720465378504,
  'recall': 0.8390831430669021,
  'f1-score': 0.8009043369625217,
  'support': 2849.0},
 'weighted avg': {'precision': 0.8551202650684995,
  'recall': 0.8055458055458056,
  'f1-score': 0.8103473247698922,
  'support': 2849.0}}

In [11]:
save_confusion_matrix_plot(
    y_true=y_test.to_numpy(),
    y_pred=test_pred,
    class_names=target_names,
    output_path=SPEECH_TRIAL_001_CONFUSION_MATRIX_PATH,
)

SPEECH_TRIAL_001_CONFUSION_MATRIX_PATH

WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/reports/figures/experiments/speech_trial_001_mfcc_random_forest_confusion_matrix.png')

In [12]:
SPEECH_TRIAL_001_PREDICTIONS_PATH.parent.mkdir(parents=True, exist_ok=True)

predictions_df = test_df.copy()
predictions_df["dysarthric_probability"] = test_score
predictions_df["predicted_label_encoded"] = test_pred
predictions_df["predicted_label"] = predictions_df["predicted_label_encoded"].map(
    SPEECH_INVERSE_LABEL_MAPPING
)
predictions_df["true_label"] = predictions_df["label_encoded"].map(
    SPEECH_INVERSE_LABEL_MAPPING
)
predictions_df["is_correct"] = (
    predictions_df["label_encoded"] == predictions_df["predicted_label_encoded"]
)

predictions_df.to_csv(SPEECH_TRIAL_001_PREDICTIONS_PATH, index=False)

SPEECH_TRIAL_001_PREDICTIONS_PATH

WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/data/processed/experiments/speech_trial_001_mfcc_random_forest_predictions.csv')

In [13]:
save_markdown_results(
    metrics=test_metrics,
    report=report,
    output_path=SPEECH_TRIAL_001_RESULTS_PATH,
    title="Speech Trial 001 MFCC Random Forest Results",
)

SPEECH_TRIAL_001_RESULTS_PATH

WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/reports/experiments/speech_trial_001_mfcc_random_forest_results.md')

In [14]:
dysarthric_recall = report["dysarthric"]["recall"]
dysarthric_f1 = report["dysarthric"]["f1-score"]
roc_auc = test_metrics["roc_auc"]

accepted = (
    dysarthric_recall >= 0.75
    and dysarthric_f1 >= 0.70
    and roc_auc >= 0.80
)

{
    "dysarthric_recall": dysarthric_recall,
    "dysarthric_f1": dysarthric_f1,
    "roc_auc": roc_auc,
    "accepted": accepted,
}

{'dysarthric_recall': 0.9470468431771895,
 'dysarthric_f1': 0.7705053852526926,
 'roc_auc': 0.9426279894010781,
 'accepted': True}

In [ ]:
if accepted:
    save_model(model, SPEECH_TRIAL_001_MODEL_PATH)
    print("Accepted speech model saved to:", SPEECH_TRIAL_001_MODEL_PATH)
else:
    print("Model rejected. Results and predictions saved, model artifact not saved.")

Accepted speech model saved to: C:\Users\Asus\Downloads\Uni_work\Grad-Project\rural-stroke-assist\models\experiments\speech\trial_001_mfcc_random_forest\model.pkl


: 

## Empty Audio Handling

During feature extraction, one audio file was found to contain zero samples despite being marked as readable in the original manifest.

Professional correction:
- empty audio files are filtered before feature extraction,
- feature extraction includes defensive error handling,
- failed files are saved to `speech_feature_extraction_failures.csv`,
- the modeling dataset is built only from successfully extracted features.